# 1. Data Acquisition and External Research

## Objective
Research and download external data sources to augment the training data for Old Assyrian (Akkadian) to English translation.

### Key External Sources Identified:
1. **ORACC (Open Richly Annotated Cuneiform Corpus)** - Major source of Akkadian texts with translations
2. **Akkademia** - Neural machine translation project with Akkadian-English parallel data
3. **eBL Dictionary** - Electronic Babylonian Library
4. **Deep Past Challenge publications.csv** - Contains ~880 scholarly PDFs with translations

In [1]:
# Parameters and Paths
import os
from pathlib import Path

OUTPUT_DIR = Path("../output/external_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("../data")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Data directory: {DATA_DIR}")

Output directory: ../output/external_data
Data directory: ../data


In [2]:
# Import necessary libraries
import requests
import pandas as pd
import json
import gzip
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## 1.1 Analyze Existing Publications Data

The publications.csv file contains OCR output from almost 900 scholarly PDFs. Let's analyze what data is available.

In [3]:
# Load and analyze publications data
publications_df = pd.read_csv(DATA_DIR / "publications.csv")
print(f"Publications shape: {publications_df.shape}")
print(f"\nColumns: {publications_df.columns.tolist()}")
print(f"\nSample page text:\n{publications_df['page_text'].iloc[0][:500]}")

Publications shape: (216602, 4)

Columns: ['pdf_name', 'page', 'page_text', 'has_akkadian']

Sample page text:
Kleine Mitteilungen\nAltassyrische Miszellen 2 — Zur Göttin Istar-ZA.AT\nVon Guido Kryszat (Mainz)\nHans Hirsch hat in seinen noch immer unersetzlichen Untersuchungen zur altassyrischen Religion auch die\ndamals bekannten Belege für eine in der Regel Istar-ZA.AT umschriebene und nur im Altassyrischen bezeugte\nGottheit zusammengestellt und diskutiert.1 Mittlerweile sind einige, wenn auch wenige, neue Belege bekannt\ngeworden. Da außerdem seinerzeit eine Be trachtung zum Umfeld der Texte, in dene


In [4]:
# Check how many have Akkadian content
print(f"Has Akkadian: {publications_df['has_akkadian'].sum()} out of {len(publications_df)}")
print(f"\nUnique PDFs: {publications_df['pdf_name'].nunique()}")

Has Akkadian: 31286 out of 216602

Unique PDFs: 952


## 1.2 Analyze Published Texts

The published_texts.csv contains transliterations without translations, but can be used to cross-reference with publications.

In [5]:
# Load published texts
published_texts_df = pd.read_csv(DATA_DIR / "published_texts.csv")
print(f"Published texts shape: {published_texts_df.shape}")
print(f"\nColumns: {published_texts_df.columns.tolist()}")
print(f"\nSample transliteration:\n{published_texts_df['transliteration'].iloc[0][:300]}")

Published texts shape: (7953, 19)

Columns: ['oare_id', 'online transcript', 'cdli_id', 'aliases', 'label', 'publication_catalog', 'description', 'genre_label', 'inventory_position', 'online_catalog', 'note', 'interlinear_commentary', 'online_information', 'excavation_no', 'oatp_key', 'eBL_id', 'AICC_translation', 'transliteration_orig', 'transliteration']

Sample transliteration:
KIŠIB a-šùr-ma-lik DUMU i-na-a a-na a-šùr-na-da DUMU <gap> ù en-um-a-šùr DUMU <gap> a-pu-tum a-na a-wa-at ṭup-pì-im iḫ-da <gap> ša ú-<gap> ḫa-sí-sú <gap>


## 1.3 Download Akkadian-English Corpus from Akkademia/HuggingFace

We will download the Akkadian-English parallel corpus from HuggingFace. This dataset was prepared by the Akkademia project and contains cleaned parallel data derived from ORACC RINAP corpora.

In [6]:
# Download Akkadian-English corpus from HuggingFace
# This dataset is derived from Akkademia/ORACC RINAP data

try:
    from datasets import load_dataset
    
    print("Loading Akkadian-English corpus from HuggingFace...")
    dataset = load_dataset("veezbo/akkadian_english_corpus", split="train")
    
    # Convert to DataFrame
    akkad_corpus_df = dataset.to_pandas()
    print(f"Loaded {len(akkad_corpus_df)} sentence pairs")
    
    # Rename columns to match our format
    akkad_corpus_df = akkad_corpus_df.rename(columns={
        'text': 'translation'
    })
    
    # Add empty transliteration column (we'll need to get the source Akkadian)
    # The HuggingFace dataset only contains English translations
    # Let's check what data is available
    print(f"\nColumns: {akkad_corpus_df.columns.tolist()}")
    print(f"\nSample translation:\n{akkad_corpus_df['translation'].iloc[0][:300]}")
    
    # Since we only have English, let's try to get the original Akkadian from the source
    # The original data is available from Akkademia GitHub
    
    akkad_corpus_df['source'] = 'huggingface_akkadian_corpus'
    akkad_corpus_df.to_csv(OUTPUT_DIR / "akkadian_english_corpus_en.csv", index=False)
    print(f"\nSaved English-only corpus to {OUTPUT_DIR / 'akkadian_english_corpus_en.csv'}")
    
except Exception as e:
    print(f"Error loading from HuggingFace: {e}")

Error loading from HuggingFace: No module named 'datasets'


In [7]:
# Download the original Akkadian text from Akkademia GitHub
# The Akkademia project has both English and Akkadian files

def download_akkademia_file(url, output_path, description="file"):
    """Download a file from Akkademia GitHub"""
    try:
        response = requests.get(url, timeout=60)
        if response.status_code == 200:
            with open(output_path, 'wb') as f:
                f.write(response.content)
            size = len(response.content)
            print(f"Downloaded {description}: {size/1024:.1f} KB")
            return True
        else:
            print(f"Failed to download {description}: Status {response.status_code}")
            return False
    except Exception as e:
        print(f"Error downloading {description}: {e}")
        return False

# Download Akkadian source file (Akkadian transliteration)
# Note: Files are train.ak (not .akk), train.en, valid.ak, valid.en
akkad_urls = [
    ("https://raw.githubusercontent.com/gaigutherz/Akkademia/master/NMT_input/train.ak", 
     OUTPUT_DIR / "akkademia_train.ak", "Akkadian source (train)"),
    ("https://raw.githubusercontent.com/gaigutherz/Akkademia/master/NMT_input/train.en", 
     OUTPUT_DIR / "akkademia_train.en", "English translation (train)"),
    ("https://raw.githubusercontent.com/gaigutherz/Akkademia/master/NMT_input/valid.ak", 
     OUTPUT_DIR / "akkademia_valid.ak", "Akkadian source (valid)"),
    ("https://raw.githubusercontent.com/gaigutherz/Akkademia/master/NMT_input/valid.en", 
     OUTPUT_DIR / "akkademia_valid.en", "English translation (valid)"),
]

for url, path, desc in akkad_urls:
    download_akkademia_file(url, path, desc)

Downloaded Akkadian source (train): 4748.0 KB
Downloaded English translation (train): 4380.3 KB
Downloaded Akkadian source (valid): 269.5 KB
Downloaded English translation (valid): 246.1 KB


In [8]:
# Parse the Akkademia parallel corpus (train and valid) and combine into a DataFrame
train_akk = OUTPUT_DIR / "akkademia_train.ak"
train_en = OUTPUT_DIR / "akkademia_train.en"
valid_akk = OUTPUT_DIR / "akkademia_valid.ak"
valid_en = OUTPUT_DIR / "akkademia_valid.en"

def load_akkademia_pair(akk_path, en_path, source_name):
    """Load and parse an Akkadian-English file pair"""
    if akk_path.exists() and en_path.exists():
        with open(akk_path, 'r', encoding='utf-8') as f:
            akk_lines = f.readlines()
        with open(en_path, 'r', encoding='utf-8') as f:
            en_lines = f.readlines()
        
        print(f"Loaded {len(akk_lines)} lines from {source_name}")
        
        parallel_data = []
        for akk, en in zip(akk_lines, en_lines):
            akk = akk.strip()
            en = en.strip()
            if akk and en and len(akk) > 5 and len(en) > 5:
                parallel_data.append({
                    'transliteration': akk,
                    'translation': en,
                    'source': source_name
                })
        return parallel_data
    else:
        print(f"Files not found for {source_name}")
        return []

# Load train and validation data
all_parallel_data = []
all_parallel_data.extend(load_akkademia_pair(train_akk, train_en, 'akkademia_train'))
all_parallel_data.extend(load_akkademia_pair(valid_akk, valid_en, 'akkademia_valid'))

if all_parallel_data:
    akkademia_df = pd.DataFrame(all_parallel_data)
    print(f"\nCreated parallel corpus with {len(akkademia_df)} sentence pairs")
    
    # Save the parallel corpus
    akkademia_df.to_csv(OUTPUT_DIR / "akkademia_parallel_corpus.csv", index=False)
    print(f"Saved to {OUTPUT_DIR / 'akkademia_parallel_corpus.csv'}")
    
    # Show sample
    print("\nSample pairs:")
    for i in range(min(3, len(akkademia_df))):
        print(f"\nAkkadian: {akkademia_df['transliteration'].iloc[i][:100]}...")
        print(f"English: {akkademia_df['translation'].iloc[i][:100]}...")
else:
    print("No Akkademia files found, skipping parallel corpus creation")

Loaded 50478 lines from akkademia_train
Loaded 2812 lines from akkademia_valid

Created parallel corpus with 40424 sentence pairs
Saved to ../output/external_data/akkademia_parallel_corpus.csv

Sample pairs:

Akkadian: 𒉭𒁄𒌀𒆠𒋗𒄣𒊒𒈾𒉘𒀭...𒀭𒊺𒊒𒀀𒀊𒁀...𒉿𒄘𒀭𒊩𒌆𒃞𒈾𒃻𒀀𒈾𒁁𒂁𒆳𒆳...𒀭𒋾𒀠𒋃...𒅕𒁍𒌑𒀀𒈾𒈗𒌑𒋾𒄊𒀴...𒈠𒈬𒍦𒊮𒅆𒃸𒈨𒌍𒀀𒈾...𒋗𒊑𒅔𒉌𒍣𒅗𒊒𒆗𒉡𒉡𒌨𒆧𒆳𒌦𒈨𒌍𒋗𒂊𒌀...𒆗𒂷𒆠......
English: Precious scion of Baltil (Aššur), beloved of the god(dess) (DN and) Šērūa, ... , creation of the god...

Akkadian: 𒄣𒊏𒁺...𒌑𒊕𒉌𒋙𒊺𒁍𒍑𒋗...𒀸𒄑𒆪𒌑𒌑𒆥𒌅...𒈬𒌓𒊑𒆪......
English: warrior ... who made ... bow down at his feet ... , who put ... to the sword (lit. “weapon”), ... ci...

Akkadian: ...𒌑𒃻𒀾𒅆𒋡𒄊𒅀𒋗...𒃻𒁲𒂊...𒃮𒇷...𒌋𒅗𒄀𒂇𒈗𒈨𒌍𒀀𒈨𒁈𒈨𒌍...𒈬𒌓𒊑𒆠...𒌑𒋧𒃲𒈝𒍢𒄿𒊒...𒁕𒄉𒈨...
English: ... he made ... kiss his feet ... mountains ... in/of battle ... he (a god) made my weapon/rule grea...


## 1.4 Try to Fetch Data from ORACC

ORACC provides JSON APIs for accessing their data. Let's explore this.

In [9]:
# Try to access ORACC API for Old Assyrian texts
# ORACC has multiple projects, let's try to access some

oracc_projects = [
    "saao",  # State Archives of Assyria
    "rinap",  # Royal Inscriptions of the Neo-Assyrian Period
    "cams",  # Corpus of Akkadian Mentions of Sources
]

def fetch_oracc_project(project):
    """Try to fetch ORACC project data"""
    url = f"http://oracc.museum.upenn.edu/{project}/json"
    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            return response.json()
    except Exception as e:
        print(f"Error fetching {project}: {e}")
    return None

# Test fetching ORACC data
for proj in oracc_projects:
    print(f"Trying ORACC project: {proj}")
    data = fetch_oracc_project(proj)
    if data:
        print(f"  Success! Keys: {list(data.keys())[:5]}")
    else:
        print(f"  Failed or no data")

Trying ORACC project: saao
Error fetching saao: HTTPSConnectionPool(host='oracc.museum.upenn.edu', port=443): Max retries exceeded with url: /saao/json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
  Failed or no data
Trying ORACC project: rinap
Error fetching rinap: HTTPSConnectionPool(host='oracc.museum.upenn.edu', port=443): Max retries exceeded with url: /rinap/json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
  Failed or no data
Trying ORACC project: cams
Error fetching cams: HTTPSConnectionPool(host='oracc.museum.upenn.edu', port=443): Max retries exceeded with url: /cams/json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
  F

## 1.5 Use Existing Resources from Data Directory

Since external APIs may not be accessible, let's extract additional data from the provided dataset.

In [10]:
# Load the OA Lexicon to understand vocabulary
lexicon_df = pd.read_csv(DATA_DIR / "OA_Lexicon_eBL.csv")
print(f"Lexicon shape: {lexicon_df.shape}")
print(f"\nColumns: {lexicon_df.columns.tolist()}")
print(f"\nSample entries:")
lexicon_df.head(5)

Lexicon shape: (39332, 9)

Columns: ['type', 'form', 'norm', 'lexeme', 'eBL', 'I_IV', 'A_D', 'Female(f)', 'Alt_lex']

Sample entries:


,type,form,norm,lexeme,eBL,I_IV,A_D,Female(f),Alt_lex
0,word,áb ša-ra-ni,ab šarrānē,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN
1,word,áb ša-ra-nim,ab šarrānem,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN
2,word,áb-ša-ra-nim,ab šarrānem,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN
3,word,áb-ša-ra-ni,ab šarrānē,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN
4,word,áb-ša-ra-ni₇,ab šarrānē,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN


In [11]:
# Load the eBL Dictionary
dictionary_df = pd.read_csv(DATA_DIR / "eBL_Dictionary.csv")
print(f"Dictionary shape: {dictionary_df.shape}")
print(f"\nColumns: {dictionary_df.columns.tolist()}")
print(f"\nSample entries:")
dictionary_df.head(3)

Dictionary shape: (19215, 3)

Columns: ['word', 'definition', 'derived_from']

Sample entries:


,word,definition,derived_from
0,-a I,"""my"" (1 sg. pron. suff.)",cf. -ī I
1,-am I,"""to me"" (1 sg. dat. suff.) 2. vent. affix (cf....",NaN
2,-atti I,(adv. endings) (cf. GAG §113l),NaN


## 1.6 Extract Translation Candidates from Publications

We'll try to find pairs of Akkadian transliteration and English translation in the publications data.

In [12]:
# Look for patterns in publications that might indicate translations
# The publications contain both the original text and translations

def extract_text_pairs(text):
    """Attempt to extract potential Akkadian-English pairs"""
    pairs = []
    
    # Look for patterns that indicate translations
    # Common patterns: "(tr.)" "translation" "cf."
    
    # Split into potential segments
    lines = text.split('\n')
    
    akkadian_pattern = re.compile(r'^[A-Z][a-z]+-[A-Za-z]+|^[0-9]+[A-Z]')
    
    for i, line in enumerate(lines[:50]):  # Check first 50 lines
        # Check for Akkadian-like content
        if len(line) > 20 and not line.startswith('http'):
            has_upper = any(c.isupper() for c in line)
            has_lower = any(c.islower() for c in line)
            has_numbers = any(c.isdigit() for c in line)
            
            if has_upper and has_lower and not has_numbers:
                pairs.append(line[:200])
    
    return pairs

# Sample a publication and look for translation content
sample_pub = publications_df[publications_df['has_akkadian'] == True].iloc[0]
print(f"Sample PDF: {sample_pub['pdf_name']}")
print(f"\nSample text:\n{sample_pub['page_text'][:1500]}")

Sample PDF: (OCR) JCS 59, 2007, pp. 93-106 - Larsen, Mogens T. - Individual and Family in Old Assyrian Society.pdf

Sample text:
94\nMOGENS TROLLE LARSEN\nINDIVIDUAL AND FAMILY IN OLD ASSYRIAN SOCIETY\n95\nand textiles; the traders more-or-less permanently\nbased in Kanesh took care of the sale of these\ncommodities, either directly on the market or by\nway of commission arrangements with traveling\nmerchants, and sums of silver (plus some gold)\nwere shipped back to Assur in order to be re-\ninvested in new caravans carrying tin and textiles.\nA very lively trade in copper and wool took place\nwithin Anatolia itself, and that too was in the\nhands of the Assyrian merchants.' The Old\nAssyrian trade in Anatolia was embedded in a\nmuch larger, interregional network that is either\npoorly or not at all attested in either archaeological\nor textual sources: Assur itself produced only some\nof the textiles sent to Anatolia and imported others\nfrom the cities of the alluvial Babylonian pla

## 1.7 Compile External Data Summary

We'll save a summary of the available data and document our findings.

In [13]:
# Create summary of external data sources
external_data_summary = {
    "sources": [
        {
            "name": "ORACC",
            "description": "Open Richly Annotated Cuneiform Corpus - major source of Akkadian texts with translations",
            "url": "https://oracc.museum.upenn.edu/",
            "status": "API access attempted - using local data"
        },
        {
            "name": "Akkademia (GitHub)",
            "description": "Akkadian-English parallel corpus from Akkademia project (RINAP data)",
            "url": "https://github.com/gaigutherz/Akkademia",
            "status": "Downloaded to output/external_data/"
        },
        {
            "name": "Akkadian-English Corpus (HuggingFace)",
            "description": "Cleaned parallel corpus from veezbo/akkadian_english_corpus",
            "url": "https://huggingface.co/datasets/veezbo/akkadian_english_corpus",
            "status": "Download attempted"
        },
        {
            "name": "Competition Publications",
            "description": "OCR output from ~880 scholarly PDFs with translations",
            "status": "Analyzed - available in data/publications.csv"
        },
        {
            "name": "Published Texts",
            "description": "~8000 transliterations without translations",
            "status": "Available in data/published_texts.csv"
        },
        {
            "name": "OA Lexicon",
            "description": "Old Assyrian vocabulary with lemmatization",
            "status": "Available in data/OA_Lexicon_eBL.csv"
        },
        {
            "name": "eBL Dictionary",
            "description": "Complete Akkadian dictionary from eBL",
            "status": "Available in data/eBL_Dictionary.csv"
        }
    ],
    "local_data_files": {
        "train.csv": "~1500 transliterations with English translations",
        "test.csv": "~4 sentences (dummy), actual test has ~4000 sentences",
        "publications.csv": "~216k lines of PDF text with translations",
        "published_texts.csv": "~8000 transliterations without translations",
        "OA_Lexicon_eBL.csv": "~39k Old Assyrian words",
        "eBL_Dictionary.csv": "~19k dictionary entries"
    },
    "downloaded_files": []
}

# Add downloaded files to summary
for f in OUTPUT_DIR.iterdir():
    if f.is_file():
        external_data_summary["downloaded_files"].append({
            "name": f.name,
            "size_kb": f.stat().st_size / 1024
        })

# Save summary
with open(OUTPUT_DIR / "external_data_summary.json", 'w') as f:
    json.dump(external_data_summary, f, indent=2)

print("External data summary saved to:", OUTPUT_DIR / "external_data_summary.json")

External data summary saved to: ../output/external_data/external_data_summary.json


In [14]:
# Save processed publications metadata for later use
publications_summary = publications_df.groupby('pdf_name').agg({
    'page': 'count',
    'has_akkadian': 'sum'
}).reset_index()
publications_summary.columns = ['pdf_name', 'num_pages', 'pages_with_akkadian']
publications_summary.to_csv(OUTPUT_DIR / "publications_summary.csv", index=False)
print(f"Publications summary saved: {len(publications_summary)} PDFs")
print(publications_summary.head())

Publications summary saved: 952 PDFs
                                            pdf_name  num_pages  \
0  (OCR) AfO 51, 2005-06, pp. 247-249 - Kryszat, ...          4   
1  (OCR) CDOG 8, 2008, pp. 109-124 - Dercksen, J....         18   
2  (OCR) JCS 59, 2007, pp. 93-106 - Larsen, Mogen...         16   
3                           27_arastirma_3-libre.pdf         36   
4                           28_arastirma_3-libre.pdf         48   

   pages_with_akkadian  
0                    0  
1                    0  
2                   10  
3                    0  
4                    0  


In [15]:
# Save lexicon for use in later stages
lexicon_df.to_csv(OUTPUT_DIR / "oa_lexicon_processed.csv", index=False)
print(f"Lexicon saved to {OUTPUT_DIR / 'oa_lexicon_processed.csv'}")

# Save dictionary for use in later stages  
dictionary_df.to_csv(OUTPUT_DIR / "ebl_dictionary_processed.csv", index=False)
print(f"Dictionary saved to {OUTPUT_DIR / 'ebl_dictionary_processed.csv'}")

Lexicon saved to ../output/external_data/oa_lexicon_processed.csv
Dictionary saved to ../output/external_data/ebl_dictionary_processed.csv


## 1.8 List External Data Files

What we've collected for augmentation:

In [16]:
# List all files in output directory
print("External data files collected:")
for f in OUTPUT_DIR.iterdir():
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

External data files collected:
  akkademia_train.en: 4380.3 KB
  publications_summary.csv: 53.8 KB
  akkademia_parallel_corpus.csv: 9944.2 KB
  akkademia_train.ak: 4748.0 KB
  ebl_dictionary_processed.csv: 1582.9 KB
  external_data_summary.json: 2.3 KB
  akkademia_valid.ak: 269.5 KB
  akkademia_valid.en: 246.1 KB
  oa_lexicon_processed.csv: 3432.5 KB


In [17]:
# Verify notebook is valid JSON
import json

notebook_path = Path("notebooks/01_data_acquisition.ipynb")
if notebook_path.exists():
    with open(notebook_path, 'r') as f:
        nb = json.load(f)
    print(f"Notebook is valid JSON with {len(nb['cells'])} cells")